Cell 1 — tìm project root và cấu hình database

In [5]:
import os
import shutil
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (
            (candidate / "dbt_project" / "dbt_project.yml").is_file()
            and (candidate / "data" / "raw").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy repository CommerceFlow360."
    )

PROJECT_ROOT = find_project_root()
DBT_DIR = PROJECT_ROOT / "dbt_project"
DUCKDB_PATH = Path(
    os.environ.get(
        "DUCKDB_PATH",
        PROJECT_ROOT / "commerceflow.duckdb"
    )
).resolve()

if not DUCKDB_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy DuckDB: {DUCKDB_PATH}. "
        "Hãy chạy notebook Bronze trước."
    )

os.environ["DUCKDB_PATH"] = str(DUCKDB_PATH)

profiles_dir = Path.home() / ".dbt"
profiles_dir.mkdir(parents=True, exist_ok=True)

profile_source = DBT_DIR / "profiles.yml.example"
profile_target = profiles_dir / "profiles.yml"

if not profile_target.exists():
    shutil.copy(profile_source, profile_target)

print("Project root:", PROJECT_ROOT)
print("DuckDB:", DUCKDB_PATH)
print("dbt profile:", profile_target)

Project root: D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360
DuckDB: D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360\commerceflow.duckdb
dbt profile: C:\Users\LENOVO\.dbt\profiles.yml


Cell 2 — hàm chạy dbt

In [6]:
def run_dbt(*args):
    command = ["dbt", *args, "--no-use-colors"]

    dbt_env = os.environ.copy()
    dbt_env["PYTHONUTF8"] = "1"
    dbt_env["PYTHONIOENCODING"] = "utf-8"

    result = subprocess.run(
        command,
        cwd=DBT_DIR,
        env=dbt_env,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"Lệnh dbt thất bại: {' '.join(command)}"
        )

    return result

Cell 3 — kiểm tra kết nối và chạy năm dimension

In [7]:
run_dbt("debug")

run_dbt(
    "run",
    "--select",
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_date",
    "dim_geography",
)

11:22:33  Running with dbt=1.12.6
11:22:33  dbt version: 1.12.6
11:22:33  python version: 3.12.6
11:22:33  python path: C:\Program Files\Python312\python.exe
11:22:33  os info: Windows-11-10.0.26300-SP0
11:22:33  Using profiles dir at C:\Users\LENOVO\.dbt
11:22:33  Using profiles.yml file at C:\Users\LENOVO\.dbt\profiles.yml
11:22:33  Using dbt_project.yml file at D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360\dbt_project\dbt_project.yml
11:22:33  adapter type: duckdb
11:22:33  adapter version: 1.11.1
11:22:33  Configuration:
11:22:33    profiles.yml file [OK found and valid]
11:22:33    dbt_project.yml file [OK found and valid]
11:22:33  Required dependencies:
11:22:33   - git [OK found]

11:22:33  Connection:
11:22:33    database: commerceflow
11:22:33    schema: main
11:22:33    path: D:\KHOI\DAI HOC\TREN TRUONG\NAM3\KI 1\DATA WAREHOUSE\DOAN\CommerceFlow360\commerceflow.duckdb
11:22:33    config_options: None
11:22:33    extensions: None
11:22:33    settin

CompletedProcess(args=['dbt', 'run', '--select', 'dim_customer', 'dim_product', 'dim_seller', 'dim_date', 'dim_geography', '--no-use-colors'], returncode=0, stdout="11:22:39  Running with dbt=1.12.6\n11:22:39  Registered adapter: duckdb=1.11.1\n11:22:39  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.\nThere are 2 unused configuration paths:\n- models.commerceflow360.gold.facts\n- models.commerceflow360.gold.marts\n11:22:40  Found 15 models, 72 data tests, 10 sources, 522 macros\n11:22:40  \n11:22:40  Concurrency: 4 threads (target='dev')\n11:22:40  \n11:22:40  1 of 5 START sql table model main.dim_customer ................................. [RUN]\n11:22:40  2 of 5 START sql table model main.dim_date ..................................... [RUN]\n11:22:40  3 of 5 START sql table model main.dim_geography ................................ [RUN]\n11:22:40  4 of 5 START sql table model main.dim_product .................................. [R

Cell 4 — kiểm tra bảng sau khi chạy

In [8]:
dimension_tables = [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_date",
    "dim_geography",
]

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as connection:
    results = []

    for table_name in dimension_tables:
        row_count = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]

        results.append({
            "table_name": table_name,
            "row_count": row_count,
        })

dimension_counts = pd.DataFrame(results)
display(dimension_counts)

assert dimension_counts["row_count"].gt(0).all(), (
    "Có dimension chưa có dữ liệu."
)

,table_name,row_count
0,dim_customer,96096
1,dim_product,32951
2,dim_seller,3095
3,dim_date,1096
4,dim_geography,19177


Cell 5 — chạy bốn fact

[Sửa] Bản trước chỉ build 5 dimension; 4 fact được chạy tay ngoài notebook nên người khác chạy lại notebook sẽ không có fact. Cell này build và chạy test cho toàn bộ tầng Gold (dimension + fact) theo thứ tự phụ thuộc của dbt.

In [ ]:
run_dbt(
    "build",
    "--select",
    "fact_orders",
    "fact_order_items",
    "fact_payments",
    "fact_reviews",
)

Cell 6 — kiểm tra số dòng fact

In [ ]:
fact_tables = ["fact_orders", "fact_order_items", "fact_payments", "fact_reviews"]

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as connection:
    fact_counts = pd.DataFrame([
        {"table_name": t, "row_count": connection.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]}
        for t in fact_tables
    ])

display(fact_counts)

assert fact_counts["row_count"].gt(0).all(), "Có fact chưa có dữ liệu."